# Evaluation and observability for LLM applications


## Creating an account on Comet.com

[Comet](https://www.comet.com/site?from=llm&utm_source=opik&utm_medium=colab&utm_content=llamaindex&utm_campaign=opik) provides a hosted version of the Opik platform, [simply create an account](https://www.comet.com/signup?from=llm&=opik&utm_medium=colab&utm_content=llamaindex&utm_campaign=opik) and grab you API Key.

> You can also run the Opik platform locally, see the [installation guide](https://www.comet.com/docs/opik/self-host/overview/?from=llm&utm_source=opik&utm_medium=colab&utm_content=llamaindex&utm_campaign=opik) for more information.

In [ ]:
# !pip install opik llama-index llama-index-agent-openai llama-index-llms-openai --upgrade --quiet

In [ ]:
import opik

opik.configure(use_local=False)

## Preparing our environment

#### Create a .env file with the following content:
> OPENAI_API_KEY=your-api-key-here

> COMET_API_KEY=your-comet-api-key-here


In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Download some sample documents

In [ ]:
import os
import requests

# Create directory if it doesn't exist
os.makedirs("./data/paul_graham/", exist_ok=True)

# Download the file using requests
url = "https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt"
response = requests.get(url, timeout=(10, 60))
with open("./data/paul_graham/paul_graham_essay.txt", "wb") as f:
    f.write(response.content)

## Simple demo of logging with Opik

In [ ]:
from opik import track

@track
def my_function(x: int) -> int:
    return x + 1

my_function(1)

## Tracking LLM calls with Opik

In [ ]:
from opik.integrations.openai import track_openai
from openai import OpenAI

openai_client = OpenAI()
openai_client = track_openai(openai_client)

prompt="Hello, world!"

response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
      {"role":"user", "content":prompt}
    ]
)

print(response.choices[0].message.content)

## Using LlamaIndex

### Configuring the Opik <> LlamaIndex integration

You can use the Opik callback directly by calling:

In [ ]:
from llama_index.core import Settings
from llama_index.core.callbacks import CallbackManager
from opik.integrations.llama_index import LlamaIndexCallbackHandler

# Set up a callback handler that will automatically log all LlamaIndex operations to Opik
opik_callback_handler = LlamaIndexCallbackHandler()

# Integrating this handler into LlamaIndex's settings
Settings.callback_manager = CallbackManager([opik_callback_handler])

Now that the callback handler is configured, all traces will automatically be logged to Opik.

## Setup a simple LLamaIndex RAG pipeline

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

We can now query the index using the `query_engine` object:

In [ ]:
response = query_engine.query("What did the author do growing up?") 
print(response)

You can now go to the Opik app to see the trace:

![LlamaIndex trace in Opik](https://raw.githubusercontent.com/comet-ml/opik/main/apps/opik-documentation/documentation/static/img/cookbook/llamaIndex_cookbook.png)

In [ ]:
str(response)

## Prepare data for evaluation

#### Load dataset and insert into Opik

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("data/test.csv")
df.head()

Create a dataset client


In [ ]:
from opik import Opik

client = Opik()
dataset = client.get_or_create_dataset(name="Test dataset")

Insert

In [ ]:
qa_pairs = [
    {"input": row["Question"], "expected_output": row["Answer"], "context": row["Context"]} 
    for _, row in df.iterrows()
]
qa_pairs[0]

In [ ]:

dataset.insert(qa_pairs)

## Evaluation

LLM application

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham").load_data()
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine()

Track it with Opik

In [ ]:
from opik import track

@track
def my_llm_application(input: str) -> str:
    response = query_engine.query(input)
    return str(response)

Track the LLM calls

In [ ]:
import openai
from opik.integrations.openai import track_openai

# Define the task to evaluate
openai_client = track_openai(openai.OpenAI())

MODEL = "gpt-3.5-turbo"

Define the evaluation task

In [ ]:
def evaluation_task(x):
    return {
        "output": my_llm_application(x['input'])
    }

Create a dataset client


In [ ]:
from opik import Opik

client = Opik()
dataset = client.get_or_create_dataset(name="Test dataset")

Define evaluation metrics

In [ ]:
from opik.evaluation.metrics import (
    Hallucination,
    AnswerRelevance,
    ContextPrecision,
    ContextRecall
)


# Define the metrics
hallucination_metric = Hallucination()
answer_relevance_metric = AnswerRelevance()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall() 

Run evaluation

In [ ]:
from opik.evaluation import evaluate

evaluation = evaluate(
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=[hallucination_metric, answer_relevance_metric, context_precision_metric, context_recall_metric],
    experiment_config={
        "model": MODEL
    }
)